# Smartprix Laptop Dataset — Data Cleaning

**Dataset:** 1,020 laptop listings scraped from [Smartprix](https://www.smartprix.com/)  
**Goal:** Clean raw scraped data into a structured, analysis-ready CSV

## Cleaning Pipeline

| Step | Description |
|------|-------------|
| 1 | Remove thin-space characters & inspect raw columns |
| 2 | Validate each column — replace invalid values with `NaN` |
| 3 | Realign misplaced values to their correct columns |
| 4 | Fix product name — strip spec string from model name |
| 5–15 | Parse & convert each column to clean format |
| 16 | Drop redundant column (`screen_feature2`) |
| 17 | Final validation & export to CSV |

## Setup

In [119]:
import pandas as pd
import numpy as np
import re

In [120]:
fd = pd.read_csv('smartprix_full_df 2.csv')
df = fd.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   name                 1020 non-null   object 
 1   price                1020 non-null   object 
 2   spec_score           1020 non-null   int64  
 3   votes                1020 non-null   object 
 4   user_rating          1020 non-null   float64
 5   os                   1020 non-null   object 
 6   utility              1020 non-null   object 
 7   thickness            1019 non-null   object 
 8   weight               910 non-null    object 
 9   warranty             756 non-null    object 
 10  screen_size          1020 non-null   object 
 11  resolution           1020 non-null   object 
 12  ppi                  1020 non-null   object 
 13  battery              957 non-null    object 
 14  screen_feature1      924 non-null    object 
 15  screen_feature2      456 non-null    o

## Step 1 — Remove Thin-Space Characters & Inspect Columns

The raw data contains Unicode thin-space (`\u2009`) characters embedded in string values.  
These are stripped first before any validation.

In [121]:
# Remove thin-space characters from all string columns
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace('\u2009', '', regex=True)

# Preview column names
for col in df.columns:
    print(col)

name
price
spec_score
votes
user_rating
os
utility
thickness
weight
warranty
screen_size
resolution
ppi
battery
screen_feature1
screen_feature2
processor_name
processor_speed
no_cores
caches
graphics_card
rom_memory
internal_memory
port_connection
wireless_connection
usb_ports
hardware_features


## Step 2 — Per-Column Validation

Each column is inspected for values that don't belong (e.g. a warranty string in the `weight` column).  
Invalid entries are replaced with `NaN` — they will be recovered in Step 3 via realignment.

In [122]:
# ── price ──
print("price — non-standard values:")
print(df.loc[~df['price'].str.contains(r'₹'), 'price'].unique())
print(f"Nulls: {df.price.isna().sum()}")

price — non-standard values:
[]
Nulls: 0


In [123]:
# ── spec_score ──
print("spec_score unique:", df.spec_score.unique())
print(f"Nulls: {df.spec_score.isna().sum()}")

spec_score unique: [70 78 56 60 43 55 59 47 64 73 51 57 53 66 71 69 46 68 74 54 62 61 65 44
 63 35 52 72 77 48 92 83 79 87 33 42 98 50 38 75 89 93 86 88 80 81 91 28
 29 30 45 19 39 34 41 37 84 32 95 90 82 23]
Nulls: 0


In [124]:
# ── votes ──
print("votes — non-standard values:")
print(df.loc[~df['votes'].str.contains(r'votes'), 'votes'].unique())
print(f"Nulls: {df.votes.isna().sum()}")

votes — non-standard values:
[]
Nulls: 0


In [125]:
# ── user_rating ──
print("user_rating unique:", df.user_rating.unique())
print(f"Nulls: {df.user_rating.isna().sum()}")

user_rating unique: [4.3 4.4 4.5 4.7 4.2 4.  4.1 4.8 4.6 3.9 3.7 3.8 3.5 3.4 3.6]
Nulls: 0


In [126]:
# ── os ──
print("os — invalid values (no 'OS' keyword):")
df.loc[~df['os'].str.contains(r'OS'), 'os'] = np.nan
print(df.os.unique())

os — invalid values (no 'OS' keyword):
['OS: Windows 11' 'OS: Mac' 'OS: DOS' 'OS: Windows 10' 'OS: Ubuntu'
 'OS: Chrome' 'OS: Linux' nan 'OS: Windows' 'OS: Windows 11 '
 'OS: DOS 3.0' 'OS: Windows 10 ']


In [127]:
# ── utility ──
df.loc[~df['utility'].str.contains(r'Utility'), 'utility'] = np.nan

In [128]:
# ── thickness ──
df.loc[~df['thickness'].str.contains(r'Thickness', na=False), 'thickness'] = np.nan

In [129]:
# ── weight ──
print("weight — non-standard values:")
print(df.loc[~df['weight'].str.contains(r'kg|gLight', na=False), 'weight'].unique())
df.loc[~df['weight'].str.contains(r'kg|gLight', na=False), 'weight'] = np.nan

weight — non-standard values:
['1 Year Warranty' nan '2 Year Warranty' '3 Year Warranty']


In [130]:
# ── resolution ──
print("resolution — non-standard values:")
print(df.loc[~df['resolution'].str.contains(r'pixels'), 'resolution'].unique())
df.loc[~df['resolution'].str.contains(r'pixels'), 'resolution'] = np.nan

resolution — non-standard values:
['~ 283PPIHighest']


In [131]:
# ── ppi ──
print("ppi — non-standard values:")
print(df.loc[~df['ppi'].str.contains('PPI'), 'ppi'].unique())
df.loc[~df['ppi'].str.contains('PPI'), 'ppi'] = np.nan

ppi — non-standard values:
['Anti Glare']


In [132]:
# ── processor_name ──
processor_pattern = 'Gen|Apple M|Intel|Qualcomm|AMD|Celeron|Kompanio|HiSilicon|Mediatek|MediaTek'
print("processor_name — non-standard values:")
print(df.loc[~df['processor_name'].str.contains(processor_pattern, case=False), 'processor_name'].unique())
df.loc[~df['processor_name'].str.contains(processor_pattern, case=False), 'processor_name'] = np.nan

processor_name — non-standard values:
['8 x (Turbo Speed upto 2GHz) Cores']


In [133]:
# ── processor_speed ──
print("processor_speed — non-standard values:")
print(df.loc[~df['processor_speed'].str.contains(r'Speed|GHz|Hz'), 'processor_speed'].unique())
df.loc[~df['processor_speed'].str.contains(r'Speed|GHz|Hz'), 'processor_speed'] = np.nan

processor_speed — non-standard values:
['4 x Performance Cores4 x Efficient Cores'
 '16GB, NVIDIA GeForce RTX 4090 GraphicsLargest'
 '12 x Performance Cores4 x Efficient Cores'
 '5 x Performance Cores6 x Efficient Cores' 'Intel Arc Graphics Graphics'
 'Octa Core' '6 x Performance Cores6 x Efficient Cores'
 '10 x Performance Cores4 x Efficient Cores' '20 Threads'
 '8GB, NVIDIA GeForce RTX 4060 GraphicsLargest' '8 x Cores']


In [134]:
# ── no_cores ──
cores_pattern = r'Threads|\(\d+P.\+.\d+E\)|Quad Core|Octa Core|Dual Core'
print("no_cores — non-standard values:")
print(df.loc[~df['no_cores'].str.contains(cores_pattern), 'no_cores'].unique())
df.loc[~df['no_cores'].str.contains(cores_pattern), 'no_cores'] = np.nan

no_cores — non-standard values:
['32GB DDR5 RAMLargest' '32GB LPDDR5X RAMLargest' '16GB RAMAverage'
 '4GB \u200eLPDDR4 RAMAverage' '24MB Cache' '16GB DDR5 RAMLargest']


In [135]:
# ── caches ──
print("caches — non-standard values:")
print(df.loc[~df['caches'].str.contains(r'Cache'), 'caches'].unique())
df.loc[~df['caches'].str.contains(r'Cache'), 'caches'] = np.nan

caches — non-standard values:
['Apple M1 Integrated Graphics Graphics' 'Apple 8 Core GPU Graphics'
 'Apple 10 Core GPU Graphics' '2TB SSD' '40 Core GPU Graphics'
 '14 Core GPU Graphics' '1TB SSD' 'Arm Mali G52 MC2 2EE Graphics Graphics'
 'Arm Mali G52 MC2 2EE Graphics' '512GB SSD' '64GB HARD DiskAverage'
 '18 Core GPU Graphics' '10 Core GPU Graphics' '30 Core GPU Graphics'
 '4GB, NVIDIA GeForce RTX 2050 GraphicsLargest'
 'Intel UHD Graphics Graphics' 'ARM Mali G72 Graphics'
 '4GB, \u200eNVIDIA GeForce RTX 3050 GraphicsAverage'
 'Intel Integrated UHD Graphics' 'Arm Mali-G72 MP3 Graphics'
 'AMD Radeon AMD Graphics' 'Integrated Intel UHD Graphics Graphics'
 'Intel Integrated Integrated Graphics'
 ' Integrated Intel UHD Graphics Graphics']


In [136]:
# ── graphics_card ──
print("graphics_card — non-standard values:")
print(df.loc[~df['graphics_card'].str.contains(r'Graphics', na=False), 'graphics_card'].unique())
df.loc[~df['graphics_card'].str.contains(r'Graphics', na=False), 'graphics_card'] = np.nan

graphics_card — non-standard values:
['8GB DDR4 RAMSmallest' '8GB RAMSmallest' '16GB RAMAverage' nan
 '48GB RAMLarge' '18GB RAMLarge' '4GB LPDDR4X RAMAverage'
 '8GB LPDDR4X RAMLargest' '18GB RAMAverage' '36GB RAMLarge'
 '36GB RAMLargest' '8GB DDR4 RAMAverage' '8GB DDR4 RAMLargest'
 '4GB LPDDR4 RAMLarge' '4GB LPDDR4 RAMAverage' '8GB DDR5 RAMAverage'
 '16GB DDR5 RAMAverage' '4GB \u200eLPDDR4 RAMAverage'
 '16GB DDR4 RAMLargest' '16GB LPDDR5 RAMLargest' '16GB LPDDR4X RAMLarge']


In [137]:
# ── rom_memory ──
print("rom_memory — non-standard values:")
print(df.loc[~df['rom_memory'].str.contains(r'RAM', na=False), 'rom_memory'].unique())
df.loc[~df['rom_memory'].str.contains(r'RAM', na=False), 'rom_memory'] = np.nan

rom_memory — non-standard values:
['256GB SSD' '512GB SSD' nan '1TB SSD' '128GB HARD DiskLarge'
 '128GB HARD DiskSmallest' '128GB SSD' '64GB HARD DiskLarge'
 '128GB HARD DiskAverage' '64GB SSD' '64GB HARD DiskAverage']


In [138]:
# ── port_connection ──
port_pattern = r'Ethernet|HDMI|Thunderbolt|Multi'
print("port_connection — non-standard values:")
print(df.loc[~df['port_connection'].str.contains(port_pattern, na=False), 'port_connection'].unique())
df.loc[~df['port_connection'].str.contains(port_pattern, na=False), 'port_connection'] = np.nan

port_connection — non-standard values:
['WiFi, Bluetooth' 'WiFi, Bluetooth v5.1' 'Bluetooth'
 'WiFi, Bluetooth v5.2' 'WiFi, Bluetooth v5.0']


In [139]:
# ── wireless_connection ──
print("wireless_connection — non-standard values:")
print(df.loc[~df['wireless_connection'].str.contains(r'Bluetooth'), 'wireless_connection'].unique())
df.loc[~df['wireless_connection'].str.contains(r'Bluetooth'), 'wireless_connection'] = np.nan

wireless_connection — non-standard values:
['3 x USB Type-C' '1 x USB 3.0, 1 x USB Type-C'
 '1 x USB 3.0, 2 x USB Type-C' '2 x USB Type-C' '1 x USB Type-C'
 '2 x USB 3.0, 2 x USB Type-C' 'Backlit Keyboard, Inbuilt Microphone']


In [140]:
# ── usb_ports ──
print("usb_ports — non-standard values:")
print(df.loc[~df['usb_ports'].str.contains(r'USB', na=False), 'usb_ports'].unique())
df.loc[~df['usb_ports'].str.contains(r'USB', na=False), 'usb_ports'] = np.nan

usb_ports — non-standard values:
['Inbuilt Microphone' '2MP Camera' 'Backlit Keyboard, Inbuilt Microphone'
 'Fingerprint Sensor, Inbuilt Microphone' nan '5MP Camera'
 'Fingerprint Sensor, Backlit Keyboard, Inbuilt Microphone']


In [141]:
# ── hardware_features ──
hw_pattern = r'Keyboard|Microphone|Camera|Sensor'
print("hardware_features — non-standard values:")
print(df.loc[~df['hardware_features'].str.contains(hw_pattern, na=False), 'hardware_features'].unique())

hardware_features — non-standard values:
[nan]


## Step 3 — Realign Misplaced Values

Scraped data often has values in the wrong column (e.g. a weight value sitting in the `thickness` column).  
This step detects which column each value truly belongs to using regex patterns, and moves it there.

In [142]:
# Regex pattern dictionary: maps each column to its expected value signature
dic = {
    'os':                  r'OS',
    'utility':             r'Utility',
    'thickness':           r'Thickness',
    'weight':              r'kg|gLight',
    'warranty':            r'Warranty',
    'resolution':          r'pixels',
    'ppi':                 r'PPI',
    'battery':             r'Battery',
    'screen_feature1':     r'Touch|Anti|Aspect',
    'processor_name':      r'Gen|Apple M|Intel|Qualcomm|AMD|Celeron|Kompanio|HiSilicon|Mediatek|MediaTek',
    'processor_speed':     r'Speed|GHz|Hz',
    'no_cores':            r'Threads|\(\d+P.\+.\d+E\)|Quad Core|Octa Core|Dual Core',
    'caches':              r'Cache',
    'graphics_card':       r'Graphics',
    'rom_memory':          r'RAM',
    'internal_memory':     r'SSD|Disk',
    'port_connection':     r'Ethernet|HDMI|Thunderbolt|Multi',
    'wireless_connection': r'Bluetooth',
    'usb_ports':           r'USB',
    'hardware_features':   r'Keyboard|Microphone|Camera|Sensor',
}

# Print column → pattern mapping
for i, (col, pattern) in enumerate(dic.items(), 1):
    print(f"{i:>2}. {col:<22} → {pattern}")

 1. os                     → OS
 2. utility                → Utility
 3. thickness              → Thickness
 4. weight                 → kg|gLight
 5. warranty               → Warranty
 6. resolution             → pixels
 7. ppi                    → PPI
 8. battery                → Battery
 9. screen_feature1        → Touch|Anti|Aspect
10. processor_name         → Gen|Apple M|Intel|Qualcomm|AMD|Celeron|Kompanio|HiSilicon|Mediatek|MediaTek
11. processor_speed        → Speed|GHz|Hz
12. no_cores               → Threads|\(\d+P.\+.\d+E\)|Quad Core|Octa Core|Dual Core
13. caches                 → Cache
14. graphics_card          → Graphics
15. rom_memory             → RAM
16. internal_memory        → SSD|Disk
17. port_connection        → Ethernet|HDMI|Thunderbolt|Multi
18. wireless_connection    → Bluetooth
19. usb_ports              → USB
20. hardware_features      → Keyboard|Microphone|Camera|Sensor


In [143]:
# Realign: for each target column, scan other columns in fd for matching values
# and copy them into df where a match is found
for target_col, pattern in dic.items():
    regex = re.compile(pattern)
    for src_col in fd.columns[5:]:
        if target_col == src_col:
            continue
        mask = fd[src_col].str.contains(regex, na=False)
        if mask.any():
            print(f"{target_col:<22} ← {src_col}")
            df.loc[mask, target_col] = fd.loc[mask, src_col]

utility                ← os
thickness              ← utility
weight                 ← utility
weight                 ← thickness
warranty               ← utility
warranty               ← thickness
warranty               ← weight
ppi                    ← resolution
screen_feature1        ← ppi
screen_feature1        ← screen_feature2
processor_name         ← processor_speed
processor_name         ← caches
processor_name         ← graphics_card
processor_speed        ← processor_name
no_cores               ← processor_name
no_cores               ← processor_speed
caches                 ← no_cores
graphics_card          ← processor_speed
graphics_card          ← caches
rom_memory             ← no_cores
rom_memory             ← graphics_card
internal_memory        ← caches
internal_memory        ← rom_memory
wireless_connection    ← port_connection
usb_ports              ← wireless_connection
hardware_features      ← wireless_connection
hardware_features      ← usb_ports


## Step 4 — Fix Product Name Column

Raw product names include the spec string in parentheses, e.g.:  
`HP Victus 15-fb0157AX Gaming Laptop (AMD Ryzen 5 5600H/ 8GB/ 512GB SSD/ Win11)`

The part before `(` is extracted as the clean model name.

In [144]:
# Preview raw names
df.name.sample(10).reset_index()

,index,name
0,726,HP 14-hr0000AU Laptop (AMD Ryzen 5 7520U/ 16GB...
1,482,HP 15s-fq3066TU Laptop (Intel Celeron N4500/ 8...
2,1011,Asus Vivobook S14 Flip 2022 TN3402QA-LZ520WS L...
3,698,Lenovo IdeaPad Gaming 3 82K101LHIN Laptop (11t...
4,547,Asus Vivobook Go 14 2023 E1404FA-NK325WS Lapto...
5,222,Chuwi Ubook X Laptop (10th Gen Core i3/ 12GB/ ...
6,491,Asus Vivobook Pro 15 M6500QC-HN751WS Laptop (A...
7,517,Lenovo IdeaPad Flex 5 82Y0004YIN Laptop (13th ...
8,40,Acer One 14 Z8-415 ‎2023 Laptop (11th Gen Core...
9,444,Lenovo V15 G3 82TTA02UIH Laptop (12th Gen Core...


In [145]:
# Split on '(' and keep only the model name
new_name = df['name'].str.split('(', expand=True)
df['name'] = new_name[0]
df.rename({'name': 'product_name'}, axis=1, inplace=True)

df[['product_name']].head()

,product_name
0,HP Victus 15-fb0157AX Gaming Laptop
1,Xiaomi Redmi G Pro 2024 Gaming Laptop
2,Tecno Megabook T1 Laptop
3,Samsung Galaxy Book2 Pro 13 Laptop
4,Apple MacBook Air 2020 MGND3HN Laptop


## Steps 5–15 — Parse & Convert Individual Columns

In [146]:
## 5. price — strip ₹ symbol and commas, cast to int
df['price'] = df['price'].str.strip('₹').str.replace(',', '').astype(np.int64)
print(df['price'].head())

0     49800
1    102990
2     23990
3     62990
4     79990
Name: price, dtype: int64


In [147]:
## 6. votes — extract number from "1,511 votes • 23 reviews"
df['votes'] = df['votes'].str.split(' ', expand=True)[0].str.replace(',', '').astype(np.int64)
print(df['votes'].head())

0       97
1       71
2      177
3     1511
4    20380
Name: votes, dtype: int64


In [148]:
## 7. os — strip "OS:" prefix
df['os'] = df['os'].str.removeprefix('OS:').str.strip()
print(f"Nulls: {df['os'].isnull().sum()}")
print(df['os'].unique())

Nulls: 5
['Windows 11' 'Mac' 'DOS' 'Windows 10' 'Ubuntu' 'Chrome' 'Linux' nan
 'Windows' 'DOS 3.0']


In [149]:
## 8. utility — strip "Utility: " prefix
df['utility'] = df['utility'].str.removeprefix('Utility: ').str.strip()
print(f"Nulls: {df['utility'].isnull().sum()}")

Nulls: 2


In [150]:
## 9. thickness — extract value after "Thickness: "
df['thickness'] = df['thickness'].str.split('Thickness: ', expand=True)[1].str.replace('mm', 'mm ')
print(df['thickness'].head())

0         23.5mm Thick
1        23.45mm Thick
2    14.8mm Ultra Slim
3          11.2mm Slim
4          16.1mm Slim
Name: thickness, dtype: object


In [151]:
## 10. weight — strip qualitative labels (Light, Heavy, etc.)
df['weight'] = df['weight'].str.replace(r'Lightest|Heaviest|Light|Average|Heavy', '', regex=True).str.strip()
print(df['weight'].head())

0    2.29kg
1     2.7kg
2    1.56kg
3      870g
4    1.29kg
Name: weight, dtype: object


In [152]:
## 11. warranty — keep only "X Year" part
df['warranty'] = df['warranty'].str.split(' ', expand=True)[[0, 1]].fillna('').agg(' '.join, axis=1).str.strip()
df.loc[df['warranty'] == '', 'warranty'] = np.nan
print(f"Nulls: {df.warranty.isna().sum()}")
print(df['warranty'].unique())

Nulls: 9
['1 Year' nan '2 Year' '3 Year']


In [153]:
## 12. screen_size — strip qualitative labels
df['screen_size'] = df.screen_size.str.replace(r'Largest|Smallest|Average|Large|Small', '', regex=True).str.strip()
print(df['screen_size'].unique())

['15.6inches' '16.1inches' '13.3inches' '13.6inches' '14inches'
 '13.4inches' '16inches' '11.6inches' '15.3inches' '13.5inches' '17inches'
 '18inches' '16.2inches' '14.2inches' '17.3inches' '14.1inches'
 '15.75inches' '12inches' '12.4inches' '15inches' '12.6inches'
 '14.5inches']


In [163]:
## 13. resolution — strip qualitative labels
df['resolution'] = df['resolution'].str.replace(r'Average|Good|Bad|Best|Poor', '', regex=True).str.replace('pixels', ' px').str.strip()
print(df['resolution'].head())

0    1920x1080 px
1    2560x1600 px
2    1920x1080 px
3    1080x1920 px
4    2560x1600 px
Name: resolution, dtype: object


In [155]:
## 14. ppi — strip "~" and label, cast to int
df['ppi'] = df['ppi'].str.strip('~ ').str.split('PPI', expand = True)[0].str.strip().astype(np.int64)

In [156]:
## 15. battery — strip "Good" label
df['battery'] = df['battery'].str.replace('Good', '', regex=False).str.strip()
print(df['battery'].head())

0    52.5Wh, 3 Cell Battery
1              80Wh Battery
2              70Wh Battery
3                       NaN
4            49.9Wh Battery
Name: battery, dtype: object


In [157]:
## 16. ram_memory — rename and strip qualitative labels
df.rename({'rom_memory': 'ram_memory'}, axis=1, inplace=True)
df['ram_memory'] = df['ram_memory'].str.replace(r'Average|Largest|Smallest|Large', '', regex=True).str.strip()
print(df['ram_memory'].head())

0       8GB DDR4 RAM
1      16GB DDR5 RAM
2     8GB LPDDR4 RAM
3    16GB LPDDR5 RAM
4      8 GB DDR4 RAM
Name: ram_memory, dtype: object


## Step 16 — Drop `screen_feature2`

`screen_feature2` has 564 null values (55% of the dataset) and is largely redundant with `screen_feature1`.  
It is dropped and `screen_feature1` is renamed to `screen_feature`.

In [158]:
df.drop(columns='screen_feature2', inplace=True)
df.rename({'screen_feature1': 'screen_feature'}, axis=1, inplace=True)
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['product_name', 'price', 'spec_score', 'votes', 'user_rating', 'os', 'utility', 'thickness', 'weight', 'warranty', 'screen_size', 'resolution', 'ppi', 'battery', 'screen_feature', 'processor_name', 'processor_speed', 'no_cores', 'caches', 'graphics_card', 'ram_memory', 'internal_memory', 'port_connection', 'wireless_connection', 'usb_ports', 'hardware_features']


## Step 17 — Final Validation

In [159]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_name         1020 non-null   object 
 1   price                1020 non-null   int64  
 2   spec_score           1020 non-null   int64  
 3   votes                1020 non-null   int64  
 4   user_rating          1020 non-null   float64
 5   os                   1015 non-null   object 
 6   utility              1018 non-null   object 
 7   thickness            771 non-null    object 
 8   weight               910 non-null    object 
 9   warranty             1011 non-null   object 
 10  screen_size          1020 non-null   object 
 11  resolution           1019 non-null   object 
 12  ppi                  1020 non-null   int64  
 13  battery              957 non-null    object 
 14  screen_feature       925 non-null    object 
 15  processor_name       1019 non-null   o

In [160]:
print("Null counts per column:")
print(df.isna().sum().sort_values(ascending = False))

Null counts per column:
thickness              249
weight                 110
screen_feature          95
battery                 63
caches                  38
processor_speed         19
port_connection         15
warranty                 9
usb_ports                6
os                       5
no_cores                 4
graphics_card            3
hardware_features        2
utility                  2
processor_name           1
resolution               1
product_name             0
price                    0
votes                    0
spec_score               0
user_rating              0
ppi                      0
screen_size              0
ram_memory               0
internal_memory          0
wireless_connection      0
dtype: int64


In [161]:
print("\nNumeric column summary:")
df[['price', 'spec_score', 'votes', 'user_rating', 'ppi']].describe().round(2)


Numeric column summary:


,price,spec_score,votes,user_rating,ppi
count,1020.00,1020.00,1020.00,1020.00,1020.00
mean,85093.11,61.15,299.34,4.32,159.20
std,69578.38,11.09,889.84,0.23,37.49
min,8000.00,19.00,51.00,3.40,100.00
25%,45467.25,54.00,80.00,4.20,141.00
50%,64240.00,61.00,101.00,4.30,141.00
75%,96992.25,68.00,177.75,4.50,162.00
max,599990.00,98.00,20380.00,4.80,323.00


## Export to CSV

In [164]:
df.to_csv('Cleaned.csv', index=True)
print(f"Saved: Cleaned.csv  |  Shape: {df.shape}")

Saved: Cleaned.csv  |  Shape: (1020, 26)
